In [101]:
# !pip install numpy
# !pip install pandas
# !pip install black

In [102]:
import numpy as numpy
import pandas as pd
from scipy import stats
import statsmodels.api as sm

# A/B Testing the Udacity Website

In these exercises, we'll be analyzing data on user behavior from an experiment run by Udacity, the online education company. More specifically, we'll be looking at a test Udacity ran to improve the onboarding process on their site.

Udacity's test is an example of an "A/B" test, in which some portion of users visiting a website (or using an app) are randomly selected to see a new version of the site. An analyst can then compare the behavior of users who see a new website design to users seeing their normal website to estimate the effect of rolling out the proposed changes to all users. While this kind of experiment has it's own name in industry (A/B testing), to be clear it's just a randomized experiment, and so everything we've learned about potential outcomes and randomized experiments apply here. 

(Udacity has generously provides the data from this test under an Apache open-source license, and you can find their [original writeup here](https://www.kaggle.com/tammyrotem/ab-tests-with-python/notebook). If you're interested in learning more on A/B testing in particular, it seems only fair while we use their data to flag they have a full course on the subject [here](https://www.udacity.com/course/ab-testing--ud257).)

## Udacity's Test

The test [is described by Udacity as follows](https://www.kaggle.com/tammyrotem/ab-tests-with-python/notebook): 

At the time of this experiment, Udacity courses currently have two options on the course overview page: "start free trial", and "access course materials".

**Current Conditions Before Change**

- If the student clicks "start free trial", they will be asked to enter their credit card information, and then they will be enrolled in a free trial for the paid version of the course. After 14 days, they will automatically be charged unless they cancel first.
- If the student clicks "access course materials", they will be able to view the videos and take the quizzes for free, but they will not receive coaching support or a verified certificate, and they will not submit their final project for feedback.

**Description of Experimented Change**

- In the experiment, Udacity tested a change where if the student clicked "start free trial", they were asked how much time they had available to devote to the course.
- If the student indicated 5 or more hours per week, they would be taken through the checkout process as usual. If they indicated fewer than 5 hours per week, a message would appear indicating that Udacity courses usually require a greater time commitment for successful completion, and suggesting that the student might like to access the course materials for free.
- At this point, the student would have the option to continue enrolling in the free trial, or access the course materials for free instead. This [screenshot](images/udacity_checkyoureready.png) shows what the experiment looks like.

**Udacity's Hope is that...**:

> this might set clearer expectations for students upfront, thus reducing the number of frustrated students who left the free trial because they didn't have enough time -- without significantly reducing the number of students to continue past the free trial and eventually complete the course. If this hypothesis held true, Udacity could improve the overall student experience and improve coaches' capacity to support students who are likely to complete the course.



## Gradescope Autograding

Please follow [all standard guidance](https://www.practicaldatascience.org/ids720_specific/autograder_guidelines.html) for submitting this assignment to the Gradescope autograder, including storing your solutions in a dictionary called `results` and ensuring your notebook runs from the start to completion without any errors.

For this assignment, please name your file `exercise_abtesting.ipynb` before uploading.

You can check that you have answers for all questions in your `results` dictionary with this code:

```python
assert set(results.keys()) == {
    "ex4_avg_oec",
    "ex5_avg_guardrail",
    "ex7_ttest_pvalue",
    "ex9_ttest_pvalue_clicks",
    "ex10_num_obs",
    "ex11_guard_ate",
    "ex11_guard_pvalue",
    "ex11_oec_ate",
    "ex11_oec_pvalue",
    "ex14_se_treatment",
}
```


### Submission Limits

Please remember that you are **only allowed THREE submissions to the autograder.** Your last submission (if you submit 3 or fewer times), or your third submission (if you submit more than 3 times) will determine your grade Submissions that error out will **not** count against this total.

## Import the Data

### Exercise 1

Begin by importing Udacity's data on user behavior [here.](https://github.com/nickeubank/MIDS_Data/tree/master/udacity_AB_testing) 

There are TWO datasets for this test — one for the control data (users who saw the original design), and one for treatment data (users who saw the experimental design). Udacity decided to show their test site to 1/2 of visitors, so there are roughly the same number of users appearing in each dataset (though this is not a requirement of AB tests).

Please rmeember to load the data directly from github to assist the autograder.

In [103]:
control_data = pd.read_csv(
    "https://github.com/nickeubank/MIDS_Data/raw/refs/heads/master/udacity_AB_testing/control_data.csv"
)
control_data.head()

,Date,Pageviews,Clicks,Enrollments,Payments
0,"Sat, Oct 11",7723,687,134.0,70.0
1,"Sun, Oct 12",9102,779,147.0,70.0
2,"Mon, Oct 13",10511,909,167.0,95.0
3,"Tue, Oct 14",9871,836,156.0,105.0
4,"Wed, Oct 15",10014,837,163.0,64.0


In [104]:
treatment_data = pd.read_csv(
    "https://github.com/nickeubank/MIDS_Data/raw/refs/heads/master/udacity_AB_testing/experiment_data.csv"
)
treatment_data.head()

,Date,Pageviews,Clicks,Enrollments,Payments
0,"Sat, Oct 11",7716,686,105.0,34.0
1,"Sun, Oct 12",9288,785,116.0,91.0
2,"Mon, Oct 13",10480,884,145.0,79.0
3,"Tue, Oct 14",9867,827,138.0,92.0
4,"Wed, Oct 15",9793,832,140.0,94.0


### Exercise 2

Explore the data. Can you identify the unit of observation of the data (e.g. what is represented by each row)?

To be clear, the columns represent stages in a user funnel:

- Some number of users arrive at the website and are counted as Pageviews,
- Some portion of those users then click to enroll (and are counted as clicks),
- Some portion of those users then actually enroll in the free trial (after seeing an informational popup, in the case of treatment individuals),
- Finally some portion of those users end up paying at the end of the free trial period.

(Note this is not the only way that A/B test data can be collected and/or reported — this is just what Udacity provided, presumably to help address privacy concerns.)

**The unit of observation of the data is website behavior by date, i.e. each day's aggregated user behavior on Udacity, with counts of pageviews, clicks, enrollments, and payments.**

## Pick your measures

### Exercise 3

The easiest way to analyze this data is to stack it into a single dataset where each observation is a day-treatment-arm (so you should end up with two rows per day, one for those who are in the treated groups, and one for those who were in the control group). Note that currently nothing in the data identifies whether a given observation is a treatment group observation or a control group observation, so you'll want to make sure to add a "treatment" indicator variable.

The variables in the data are:

- Pageviews: number of unique users visiting homepage
- Clicks: number of those users clicking "Start Free Trial"
- Enrollments: Number of people enrolling in trial
- Payments: Number of people who eventually pay for the service. Note the `payment` column reports payments for the users who first visited the site on the reported date, not payments occurring on the reported date.

In [105]:
control_data["treatment"] = 0
treatment_data["treatment"] = 1

data = pd.concat([control_data, treatment_data])
data

,Date,Pageviews,Clicks,Enrollments,Payments,treatment
0,"Sat, Oct 11",7723,687,134.0,70.0,0
1,"Sun, Oct 12",9102,779,147.0,70.0,0
2,"Mon, Oct 13",10511,909,167.0,95.0,0
3,"Tue, Oct 14",9871,836,156.0,105.0,0
4,"Wed, Oct 15",10014,837,163.0,64.0,0
...,...,...,...,...,...,...
32,"Wed, Nov 12",10042,802,NaN,NaN,1
33,"Thu, Nov 13",9721,829,NaN,NaN,1
34,"Fri, Nov 14",9304,770,NaN,NaN,1
35,"Sat, Nov 15",8668,724,NaN,NaN,1


In [106]:
data.sort_values("Date", inplace=True)
data

,Date,Pageviews,Clicks,Enrollments,Payments,treatment
34,"Fri, Nov 14",9192,735,NaN,NaN,0
34,"Fri, Nov 14",9304,770,NaN,NaN,1
27,"Fri, Nov 7",9424,781,NaN,NaN,0
27,"Fri, Nov 7",9272,767,NaN,NaN,1
6,"Fri, Oct 17",9088,780,127.0,44.0,1
...,...,...,...,...,...,...
4,"Wed, Oct 15",9793,832,140.0,94.0,1
11,"Wed, Oct 22",9947,838,162.0,92.0,0
11,"Wed, Oct 22",9737,801,128.0,70.0,1
18,"Wed, Oct 29",9262,727,201.0,96.0,1


### Exercise 4

Given Udacity's goals, what outcome are they hoping will be impacted by their manipulation?

Or, to ask the same question in the language of the Potential Outcomes Framework, what is their $Y$?

Or to ask the same question in the language of Kohavi, Tang and Xu, what is their *Overall Evaluation Criterion (OEC)*?

(I'm only asking one question, I'm just trying to phrase it using different terminologies we've encountered to help you see how they all fit together)

When you feel like you have your answer, please compute it. Store the average value of the variable in `results` under the key `ex4_avg_oec`. **Please round your answer to 4 decimal places.**

NOTE: You'll probably notice you have two choices to make when it comes to actually computing the OEC. 

- You could probably imagine either computing a ratio or a difference of two things — please calculate the difference.
- You may also be unsure whether to normalize by `Clicks`. Normalizing by clicks will help account for variation that comes from day-to-day variation in users, so it's a good thing to do. With infinite data, you'd expect to get the same results without normalizing by `Clicks` (since on average the same share of users are in each arm of the experiment), but for finite data it's a good strategy. Note that this is only ok because users make the choice to click or not *before* they see different versions of the website (it is "pre-treatment").

Just to make sure you're on track, your measure should have an average value of *about* 9%.

**The OEC is the frustrated users, i.e the number of enrollments - number of users for payments.**

In [107]:
# get frustrated users per clicks, so enrollments - payments / clicks

# data['payment_per_click'] = data['Enrollments'] - data['Payments'] / data['Clicks']
results = {}
frustrated_per_click = (
    (data["Enrollments"] - data["Payments"]) / data["Clicks"]
).mean()
results["ex4_avg_oec"] = round(frustrated_per_click, 4)
results

{'ex4_avg_oec': np.float64(0.0941)}

### Exercise 5

Given Udacity's goals, what outcome are they hoping will *not* be impacted by their manipulation? In other words, what do they want to measure to ensure their treatment doesn't have unintended negative consequences that might be really costly to their operation?

Note that while this isn't how Kohavi, Tang, and Xu use the term "guardrail metrics" — they usually use the term to refer to things we measure to ensure the experiment is working the way it should — some people would also use the term "guardrail metrics" for something that could be impacted even if the experiment is working correctly, but which the organization wants to track to ensure they aren't impacted because they are deemed really important.

Again, please normalize by `Clicks`. Store the average value of this guardrail metric as `ex5_avg_guardrail` and **round your answer to 4 decimal places.**

**The guardrail metric Udacity does not want to decrease is payments.**

In [ ]:
# payments per clicks
data["payment_per_click"] = data["Payments"] / data["Clicks"]
results["ex5_avg_guardrail"] = round(data["payment_per_click"].mean(), 4)
results

{'ex4_avg_oec': np.float64(0.0941), 'ex5_avg_guardrail': np.float64(0.1158)}

## Validating The Data

### Exercise 6

Whenever you are working with experimental data, the first thing you want to do is verify that users actually were randomly sorted into the two arms of the experiment. In this data, half of users were supposed to be shown the old version of the site and half were supposed to see the new version.

`Pageviews` tells you how many unique users visited the welcome site we are experimenting on. `Pageviews` is what is sometimes called an "invariant" or "guardrail" variable, meaning that it shouldn't vary across treatment arms—after all, people have to visit the site before they get a chance to see the treatment, so there's no way that being assigned to treatment or control should affect the number of pageviews assigned to each group.

"Invariant" variables are also an example of what are known as a "pre-treatment" variable, because pageviews are determined before users are manipulated in any way. That makes it analogous to gender or age in experiments where you have demographic data—a person's age and gender are determined before they experience any manipulations, so the value of any pre-treatment attributes should be the same across the two arms of our experiment. This is what we've previously called "checking for balance," If pre-treatment attributes aren't balanced, then we may worry our attempt to randomly assign people to different groups failed.  Kohavi, Tang and Xu call this a "trust-based guardrail metric" because it helps us determine if we should trust our data.

To test the quality of the randomization, calculate the average number of pageviews for the treated group and for the control group. Do they look similar?


In [109]:
# average number of pagevies for treated group and for control group
data.groupby("treatment")["Pageviews"].mean()

treatment
0    9339.000000
1    9315.135135
Name: Pageviews, dtype: float64

The average number of pageviews for the treated and the control group have very similar values (9339 and 9315 respectively), so the pre treatment attriubtes are in fact balanced.

### Exercise 7

"Similar" is a tricky concept -- obviously, we expect *some* differences across groups since users were *randomly* divided across treatment arms. The question is whether the differences between groups are larger than we'd expect to emerge given our random assignment process. To evaluate this, let's use a `ttest` to test the statistical significance of the differences we see. 

**Note**: Remember that scipy functions don't accept `pandas` objects, so you use a scipy function, you have to pass the numpy vectors underlying your data with the `.values` operator (e.g. `df.my_column.values`). 

Does the difference in `pageviews` look statistically significant?

Store the resulting p-value in `ex7_ttest_pvalue` **rounded to four decimal places.**

In [110]:
# t test for pageviews between treatment and control group
control_pageviews = data[data["treatment"] == 0]["Pageviews"]
treatment_pageviews = data[data["treatment"] == 1]["Pageviews"]
t_stat, ex7_ttest_pvalue = stats.ttest_ind(
    control_pageviews.values, treatment_pageviews.values
)
results["ex7_ttest_pvalue"] = round(ex7_ttest_pvalue, 4)
results

{'ex4_avg_oec': np.float64(0.0941),
 'ex5_avg_guardrail': np.float64(0.1158),
 'ex7_ttest_pvalue': np.float64(0.8877)}

**The p value for the differences between the treatment and control groups for pages per views is not statistically significant (0.888), so we do not believe there are differences between the groups other than that due to randomization.**

### Exercise 8

`Pageviews` is not the only "pre-treatment" variable in this data we can use to evaluate balance/use as a guardrail metric. What other measure is pre-treatment? Review the description of the experiment if you're not sure.

The other pre-treatment variable is clicks because the users decide whether to click before seeing the treatment, so the clicking behavior should not be affected by treatment. 

### Exercise 9

Check if the other pre-treatment variable is also balanced. Store the p-value of your test of difference in `results` under the key `"ex9_ttest_pvalue_clicks"` **rounded to four decimal places.**

In [111]:
# t test for clicks between treatment and control group
control_clicks = data[data["treatment"] == 0]["Clicks"]
treatment_clicks = data[data["treatment"] == 1]["Clicks"]
t_stat, ex9_ttest_pvalue = stats.ttest_ind(
    control_clicks.values, treatment_clicks.values
)
results["ex9_ttest_pvalue_clicks"] = round(ex9_ttest_pvalue, 4)
results

{'ex4_avg_oec': np.float64(0.0941),
 'ex5_avg_guardrail': np.float64(0.1158),
 'ex7_ttest_pvalue': np.float64(0.8877),
 'ex9_ttest_pvalue_clicks': np.float64(0.9264)}

**The p value is 0.92 which is not statistically significant, so there is no significant difference in the clicks between the treatment and control groups.**

## Estimating the Effect of Experiment

### Exercise 10

Now that we've validated our randomization, our next task is to estimate our treatment effect. First, though, there's an issue with your data you've been able to largely ignore until now, but which you should get a grip on before estimating your treatment effect — can you tell what it is and what you should do about it?

Store the number of observations in your data *after* you've addressed this in `ex10_num_obs` (this is mostly meant as a way to sanity check your answer with autograder).

In [112]:
clean = data.dropna(subset=["Enrollments", "Payments"])
results["ex10_num_obs"] = len(clean)
results

{'ex4_avg_oec': np.float64(0.0941),
 'ex5_avg_guardrail': np.float64(0.1158),
 'ex7_ttest_pvalue': np.float64(0.8877),
 'ex9_ttest_pvalue_clicks': np.float64(0.9264),
 'ex10_num_obs': 46}


### Exercise 11

Now that we've established we have good balance (meaning we think randomization was likely successful), we can evaluate the effects of the experiment. Test whether the OEC and the metric you *don't* want affected have different average values in the control group and treatment group. 

Because we've randomized, this is a consistent estimate of the Average Treatment Effect of Udacity's website change.

Calculate the difference in means in your OEC and guardrail metrics using a simple t-test. Store the resulting effect estimates in `ex11_oec_ate` and `ex11_guard_ate` and p-values in `ex11_oec_pvalue` and `ex11_guard_pvalue`. **Please round all answers to 4 decimal places.** Report your ATE in *percentage points*, where `1` denotes 1 percentage point.


In [113]:
# oec = total payments / total clicks
# metric we don't want is enrollments / clicks
clean["oec"] = (clean["Enrollments"] - clean["Payments"]) / clean["Clicks"]

clean["guard"] = clean["Payments"] / clean["Clicks"]

# t test for oec
oec_t = clean[clean["treatment"] == 1]["oec"].values
oec_c = clean[clean["treatment"] == 0]["oec"].values

guard_t = clean[clean["treatment"] == 1]["guard"].values
guard_c = clean[clean["treatment"] == 0]["guard"].values

# compute ate between treatment and control for oec and guardrail
results["ex11_oec_ate"] = round((oec_t.mean() - oec_c.mean()) * 100, 4)
results["ex11_guard_ate"] = round((guard_t.mean() - guard_c.mean()) * 100, 4)

# t test
t_stat_oec, p_value_oec = stats.ttest_ind(oec_t, oec_c)
t_stat_guard, p_value_guard = stats.ttest_ind(guard_t, guard_c)
results["ex11_oec_pvalue"] = round(p_value_oec, 4)
results["ex11_guard_pvalue"] = round(p_value_guard, 4)
results

{'ex4_avg_oec': np.float64(0.0941),
 'ex5_avg_guardrail': np.float64(0.1158),
 'ex7_ttest_pvalue': np.float64(0.8877),
 'ex9_ttest_pvalue_clicks': np.float64(0.9264),
 'ex10_num_obs': 46,
 'ex11_oec_ate': np.float64(-1.5888),
 'ex11_guard_ate': np.float64(-0.4897),
 'ex11_oec_pvalue': np.float64(0.1319),
 'ex11_guard_pvalue': np.float64(0.5928)}

### Exercise 12

Do you feel that Udacity achieved their goal? Did their intervention cause them any problems? If they asked you "What would happen if we rolled this out to everyone?" what would you say?

As you answer this question, a small additional question: up until this point you've (presumably) been reporting the default p-values from the tools you are using. These, as you may recall from stats 101, are two-tailed p-values. Do those seem appropriate for your OEC?

**Based on the experiment evidence, the ATE (-0.0159) for the OEC shows that Udacity did reduce frustration in users slightly but the effect is not statistically significant (p value 0.13), so we can't say the decrease was due to their change except for due to possible randomness. The guardrail metric shows the intervention slightly reduced payments (ATE -0.0049) but still that effect was not statistically significant (p value of 0.13). Udacity was hoping to reduce frustration in users with this change without reducing people with payments, so this could be concerning, however since not significant they did technically achieve their goal. If asked about whether rolling out to everyone, we would say that there was a small reduction in frustrated users but also a slight drop in payments, but there is not statistical significance and since the guardrail metric slightly decreased, rolling this change out could still be risky with a weak effect or without strong clear improvement.**

**A one tailed test might be more appropriate for our problem since Udacity is likely cares more about whether the change will decrease frustrated users, and would take action based on that. However, because the p value was so far from significance, changing to one-tailed would not change the overall conclusion.**

### Exercise 13

One of the magic things about experiments is that all you have to do is compare averages to get an average treatment effect. However, you *can* do other things to try and increase the statistical power of your experiments, like add controls in a linear regression model. 

As you likely know, a bivariate regression is exactly equivalent to a t-test, so let's start by re-estimating the effect of treatment on your OEC using a linear regression. Can you replicate the results from your t-test? They shouldn't just be close—they should be numerically equivalent (i.e. exactly the same to the limits of floating point number precision). 

In [114]:
# oec - payments per click
# clean["oec_ppc"] = clean["Payments"] / clean["Clicks"]
clean["oec_bv"] = (clean["Enrollments"] - clean["Payments"]) / clean["Clicks"]

# regression: oec_bv ~ const + treatment
X = sm.add_constant(clean["treatment"].astype(float))
y = clean["oec_bv"].astype(float)

model = sm.OLS(y, X).fit()

ex13_oec_ate = round(model.params["treatment"] * 100, 4)
ex13_oec_pvalue = round(model.pvalues["treatment"], 4)

print("ATE (oec) from regression:", ex13_oec_ate)
print("p-value (oec) from regression:", ex13_oec_pvalue)

assert (
    ex13_oec_ate == results["ex11_oec_ate"]
), "ATE from regression does not match t-test result"
assert (
    ex13_oec_pvalue == results["ex11_oec_pvalue"]
), "p-value from regression does not match t-test result"

ATE (oec) from regression: -1.5888
p-value (oec) from regression: 0.1319


In [115]:
model.summary()
print("Standard Error of model: ", model.bse["treatment"], 4)

Standard Error of model:  0.010350094854414188 4


### Exercise 14

Now add indicator variables for the date of each observation. Do the standard errors on your `treatment` variable change? If so, in what direction?

**After adding the indicator variables for the dates, the standard errors decreased on the treatment variable from 0.01035 to 0.0066, which is about 36% decrease.**

Store your new standard error in `ex14_se_treatment`. Round your answer to 4 decimal places.

You should have found that your standard errors decreased by about 30\%—this is why, although just comparing means *works*, if you have additional variables adding them to your analysis can be helpful (all the usual rules for model specification apply — for example, you still want to be careful about overfitting, which one could argue is maybe part of what's happening here). 

In many other cases, the effect of adding controls is likely to be larger — the date indicators we added to our data are perfectly balanced between treatment and control, so we aren't adding a lot of data to the model by adding them as variables. They're accounting for some day-to-day variation (presumably in the types of people coming to the site), but they aren't controlling for any residual baseline differences the way a control like "gender" or "age" might (since those kind of individual-level attributes will never be perfectly balanced across treatment and control). 

In [116]:
X = pd.get_dummies(clean[["treatment", "Date"]], drop_first=True)
X = sm.add_constant(X).astype(float)

y = clean["oec_bv"].astype(float)

model_se = sm.OLS(y, X).fit()

# standard error on treatment
results["ex14_se_treatment"] = round(model_se.bse["treatment"], 4)

results["ex14_se_treatment"]

np.float64(0.0066)

In [117]:
# calcualte percent decrease from 0.01035 to 0.0066
print("Percent decrease in OEC:", round(((0.01035 - 0.0066) / 0.01035) * 100, 4))

Percent decrease in OEC: 36.2319


In [118]:
model_se.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 oec_bv   R-squared:                       0.806
Model:                            OLS   Adj. R-squared:                  0.602
Method:                 Least Squares   F-statistic:                     3.962
Date:                Sat, 31 Jan 2026   Prob (F-statistic):           0.000978
Time:                        15:13:45   Log-Likelihood:                 126.29
No. Observations:                  46   AIC:                            -204.6
Df Residuals:                      22   BIC:                            -160.7
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
====================================================================================
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.1079      0.016      6.651      0.000       0.074       0.142
treatment           -0.0159      0.007     -2.398      0.025      -0.030      -0.002
Date_Fri, Oct 24     0.0445      0.022      1.983      0.060      -0.002       0.091
Date_Fri, Oct 31    -0.0074      0.022     -0.331      0.744      -0.054       0.039
Date_Mon, Oct 13    -0.0231      0.022     -1.026      0.316      -0.070       0.024
Date_Mon, Oct 20    -0.0285      0.022     -1.270      0.217      -0.075       0.018
Date_Mon, Oct 27     0.0328      0.022      1.458      0.159      -0.014       0.079
Date_Sat, Nov 1     -0.0235      0.022     -1.047      0.306      -0.070       0.023
Date_Sat, Oct 11    -0.0017      0.022     -0.074      0.941      -0.048       0.045
Date_Sat, Oct 18    -0.0438      0.022     -1.950      0.064      -0.090       0.003
Date_Sat, Oct 25    -0.0309      0.022     -1.375      0.183      -0.077       0.016
Date_Sun, Nov 2      0.0549      0.022      2.441      0.023       0.008       0.101
Date_Sun, Oct 12    -0.0347      0.022     -1.542      0.137      -0.081       0.012
Date_Sun, Oct 19    -0.0178      0.022     -0.791      0.437      -0.064       0.029
Date_Sun, Oct 26    -0.0222      0.022     -0.989      0.333      -0.069       0.024
Date_Thu, Oct 16    -0.0228      0.022     -1.016      0.321      -0.069       0.024
Date_Thu, Oct 23    -0.0046      0.022     -0.203      0.841      -0.051       0.042
Date_Thu, Oct 30     0.0588      0.022      2.618      0.016       0.012       0.105
Date_Tue, Oct 14    -0.0417      0.022     -1.855      0.077      -0.088       0.005
Date_Tue, Oct 21    -0.0059      0.022     -0.260      0.797      -0.052       0.041
Date_Tue, Oct 28    -0.0287      0.022     -1.276      0.215      -0.075       0.018
Date_Wed, Oct 15    -0.0132      0.022     -0.588      0.562      -0.060       0.033
Date_Wed, Oct 22    -0.0220      0.022     -0.980      0.338      -0.069       0.025
Date_Wed, Oct 29     0.0466      0.022      2.076      0.050    4.77e-05       0.093
==============================================================================
Omnibus:                        3.871   Durbin-Watson:                   2.907
Prob(Omnibus):                  0.144   Jarque-Bera (JB):                3.826
Skew:                          -0.000   Prob(JB):                        0.148
Kurtosis:                       4.413   Cond. No.                         27.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [119]:
ex14_oec_ate = round(model_se.params["treatment"] * 100, 4)
ex14_oec_pvalue = round(model_se.pvalues["treatment"], 4)
print("ATE (oec) from regression with date Fixed Effect:", ex14_oec_ate)
print("p-value (oec) from regression with date Fixed Effect:", ex14_oec_pvalue)

ATE (oec) from regression with date Fixed Effect: -1.5888
p-value (oec) from regression with date Fixed Effect: 0.0254


### Exercise 15

Does this result have any impact on the recommendations you would offer Udacity?

In [120]:
# run again with guardrail as outcome to see effect
clean["guard_bv"] = clean["Payments"] / clean["Clicks"]
X = pd.get_dummies(clean[["treatment", "Date"]], drop_first=True)
X = sm.add_constant(X).astype(float)
y_guard = clean["guard_bv"].astype(float)

model_guard = sm.OLS(y_guard, X).fit()

ex14_guard_ate = round(model_guard.params["treatment"] * 100, 4)
ex14_guard_pvalue = round(model_guard.pvalues["treatment"], 4)
print("ATE (guardrail) from regression with date Fixed Effect:", ex14_guard_ate)
print("p-value (guardrail) from regression with date Fixed Effect:", ex14_guard_pvalue)

ATE (guardrail) from regression with date Fixed Effect: -0.4897
p-value (guardrail) from regression with date Fixed Effect: 0.4615


**Using standard errors and using the date fixed effects leaves the estimated treatment effect the same as -1.1588% but now the effect is statistically significant (p value 0.0254). After controlling for time specific variation, the treatment shows evidence of decreasing frustrated users so we would tell Udacity that we are more confident that intervention improves the student experience. However, the rollout still depends on the payment impact and we should look at the guardrails again, to see if payments are not significantly harmed. From the code above to look at the guardrail effect, we see that there is 0.4897 drop in payments per click which is not a statistially significant change (p value 0.4615).**

**Thus, with date fixed effects, there is statistically significant evidence that the intervention reduces frustrated users, and no statistically significant evidence that it harms payments per click. Given this, we would recommend a cautious rollout such as a staged or partial deployment while still closely monitoring the revenue related metrics. Once we control for day to day fluctuations in user behavior, the treatment effect becomes clearer and more precisely estimated.**

### Submission Check ###

In [123]:
assert set(results.keys()) == {
    "ex4_avg_oec",
    "ex5_avg_guardrail",
    "ex7_ttest_pvalue",
    "ex9_ttest_pvalue_clicks",
    "ex10_num_obs",
    "ex11_guard_ate",
    "ex11_guard_pvalue",
    "ex11_oec_ate",
    "ex11_oec_pvalue",
    "ex14_se_treatment",
}

In [124]:
results

{'ex4_avg_oec': np.float64(0.0941),
 'ex5_avg_guardrail': np.float64(0.1158),
 'ex7_ttest_pvalue': np.float64(0.8877),
 'ex9_ttest_pvalue_clicks': np.float64(0.9264),
 'ex10_num_obs': 46,
 'ex11_oec_ate': np.float64(-1.5888),
 'ex11_guard_ate': np.float64(-0.4897),
 'ex11_oec_pvalue': np.float64(0.1319),
 'ex11_guard_pvalue': np.float64(0.5928),
 'ex14_se_treatment': np.float64(0.0066)}